In [ ]:
import dai
import pandas as pd

def main(datasource, start_date, end_date):
    # 1. 使用filters参数读取数据
    data = dai.query(
        "SELECT date, instrument, close FROM bigalpha_2026_stock_bar1m",
        filters={"date": [start_date, end_date]}
    )
    df = data.df()
    
    # 2. 确保时间格式正确
    df['date'] = pd.to_datetime(df['date'])
    df['time'] = df['date'].dt.time

    # 3. 定义区间
    morning_end = pd.Timestamp('09:45:00').time()
    afternoon_start = pd.Timestamp('14:45:00').time()

    # 4. 计算早盘和尾盘平均价
    morning_avg = df[(df['time'] > pd.Timestamp('09:30:00').time()) & 
                    (df['time'] <= morning_end)].groupby(['date', 'instrument'])['close'].mean().rename('morning_avg')
    
    afternoon_avg = df[(df['time'] >= afternoon_start) & 
                      (df['time'] < pd.Timestamp('15:00:00').time())].groupby(['date', 'instrument'])['close'].mean().rename('afternoon_avg')

    # 5. 合并
    factor_df = pd.merge(morning_avg, afternoon_avg, on=['date', 'instrument'])
    factor_df['factor'] = factor_df['afternoon_avg'] / factor_df['morning_avg']

    # 6. 返回标准格式
    return factor_df[['date', 'instrument', 'factor']].reset_index(drop=True)